# Differential measurements


```{admonition} How To Work
:class: important
Complete the **Preparatory Homework (BAS)** part of this manual **individually**.

Complete the **Practicum (IC)** part **in pairs**. Divide the work concretely: one student builds the left current-to-voltage converter while the other builds the right one, then swap roles for the measurements and checks.

Download the fillable entry from the page download menu under **Log book**. Fill in Preparation before class and the measurement tables as you work. Keep your predictions, LTspice plots, calculations, and short explanations in your log book entry. You will compare these records with your measurements during the practicum and use them in your onepager report.
```

```{typst} 5.2_differential_measurements_logbook.typ
:label: Entry 5.2
:fillable:
```


## Preparatory Homework


### Background
```{admonition} Preparation
:class: tip
Before starting this experiment, make sure that you are familiar with the following introductory manuals:

* [ALPACA 2: programming and the host connection](../intros/alpaca2/programming.ipynb)
* [ALPACA 2: generating and acquiring signals](../intros/alpaca2/signals.ipynb)
* [Breadboard](../intros/breadboard.ipynb)
* [Oscilloscope](../intros/oscilloscope.ipynb)

For the simulation part of this experiment, follow the [LTspice tutorial](../intros/ltspice.ipynb).

Revisit [Manual 4.1](../week4/4.1_opamp_basics.ipynb) in case you do not remember:

* how the non-inverting and inverting amplifier gains follow from the virtual short (Tasks A1 and A3 of Manual 4.1);
* how a current-to-voltage converter turns a photodiode current into a voltage, and why the output goes negative (Tasks A5, S3, and I5 of Manual 4.1).

Read the textbook sections on the current-to-voltage converter and the photodiode amplifier in [textbook chapter 7](../../../theory_part/7_opamp1.md) and on differential amplifiers in [textbook chapter 8](../../../theory_part/8_opamp2.md).
```

<span style="background:#ff1493;color:#fff;padding:2px 8px;border-radius:4px;font-weight:700;font-size:0.85em;box-shadow:0 0 6px rgba(255,20,147,0.6);">&#x1F6A9; TODO: replace the two textbook links above with the published textbook URLs or anchors for chapters 7 and 8</span>

#### The question this manual answers

You build a detector that points at a moving light. Two photodiodes sit side by side with a small piece of opaque card between them, so the left photodiode sees a torch on the left better than the right photodiode does, and the other way around. Electronics turn the two photocurrents into one voltage, the ALPACA reads that voltage, and your software shows an arrow to the side where the light is.

```{figure} images/sensor_pair_topview.svg
---
name: fig-sensor-pair-topview
height: 360px
---
The sensor pair seen from above. The card divider shades each photodiode from light on the far side.
```

```{figure} images/detector_block_diagram.svg
---
name: fig-detector-block-diagram
height: 300px
---
The moving light detector as blocks. Each photodiode current becomes a voltage first; only their difference is amplified and digitised.
```

#### Why one photodiode cannot answer it

A single photodiode with a current-to-voltage converter, as in Task I5 of Manual 4.1, tells you how much light it receives. It cannot tell you where the light is. If its output grows, the torch may have moved towards it, or someone may have switched on the room lights. The room light reaches both photodiodes almost equally, while a torch on one side reaches mainly one of them. The direction is therefore in the **difference** of the two signals, and the room light is in the part the two signals **share**. This manual is about building a circuit that keeps the first part and rejects the second.

#### Every pair of voltages has a difference part and a common part

Call the two converter outputs $U_L$ (left) and $U_R$ (right). Any such pair can be rewritten as a differential voltage and a common-mode voltage:

$$
U_d = U_R - U_L, \qquad U_c = \frac{U_R + U_L}{2}
$$

A differential amplifier is built to amplify $U_d$ and to ignore $U_c$. A real one does not ignore $U_c$ completely, so its output is:

$$
U_{out} = A_d \, U_d + A_c \, U_c
$$

with $A_d$ the differential gain and $A_c$ the common-mode gain. How well the amplifier separates the two is expressed by the common-mode rejection ratio, $\mathrm{CMRR} = |A_d / A_c|$, often quoted in decibels as $20 \log_{10}(\mathrm{CMRR})$. The derivations behind these definitions are in textbook chapter 8; this manual measures all three quantities for a circuit you build yourself.

#### The circuit in one picture

```{figure} images/dual_tia_subtractor.svg
---
name: fig-dual-tia-subtractor
height: 400px
---
The full detector front end. Two current-to-voltage converters (feedback resistor $R_f$, photodiode cathode at the virtual ground, as in Manual 4.1) feed a four-resistor differential amplifier. Opamp supply pins ($\pm 12\,\mathrm{V}$, pins 4 and 8) are not drawn, but you must wire them for every opamp, as in Manual 4.1.
```

Each converter is the circuit from Task I5 of Manual 4.1. With separate feedback resistances, $U_L = -I_L R_{f,L}$ and $U_R = -I_R R_{f,R}$: more light gives a more negative voltage. The third opamp with resistors $R_1$ to $R_4$ is the only new block. You derive its behaviour in [Task A2](Task_A2_5_2), find out what resistor mismatch does to it in [Task A3](Task_A3_5_2), and shift its output into the ADC window in [Task A4](Task_A4_5_2).

#### Balance the sensor channels before subtracting them

Two photodiodes can produce different currents under the same illumination. To balance them, you need $I_L R_{f,L} = I_R R_{f,R}$. Adjust a converter's feedback resistance to correct this gain mismatch. Keep the four subtractor resistors matched, because their ratios set common-mode rejection. You test which resistance to adjust in [Task A1](Task_A1_5_2), then add a trimmer in [Task I1](Task_I1_5_2).

Unless a task specifies a mismatch, use $R_{f,L} = R_{f,R} = 100\,\mathrm{k}\Omega$ for the homework. A software baseline removes the remaining offset at one light level. Checking the balance at a second light level tests whether you also corrected the gain mismatch.


(Task_B1_5_2)=
#### Task B1: Split each pair into difference and common part
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

The table below shows converter outputs for four situations. For each row, calculate $U_d = U_R - U_L$ and $U_c = (U_R + U_L)/2$, and note which part changes.

| situation | $U_L$ | $U_R$ | $U_d$ | $U_c$ |
|---|---|---|---|---|
| room light only | $-0.50\,\mathrm{V}$ | $-0.50\,\mathrm{V}$ |  |  |
| torch on the left | $-1.50\,\mathrm{V}$ | $-0.50\,\mathrm{V}$ |  |  |
| torch on the right | $-0.50\,\mathrm{V}$ | $-1.50\,\mathrm{V}$ |  |  |
| brighter room light, no torch | $-1.50\,\mathrm{V}$ | $-1.50\,\mathrm{V}$ |  |  |

1. Which rows would a single left photodiode confuse with each other? Look at the $U_L$ column only.
2. Which quantity, $U_d$ or $U_c$, tells the software which side the torch is on? Which quantity tells it how bright the room is?

Write the completed table and your two answers in your log book entry. You will reuse these exact numbers in [Task A1](Task_A1_5_2).

```{admonition} Check your answer
:class: answer, dropdown
Room light only: $U_d = 0$, $U_c = -0.50\,\mathrm{V}$. Torch left: $U_d = +1.00\,\mathrm{V}$, $U_c = -1.00\,\mathrm{V}$. Torch right: $U_d = -1.00\,\mathrm{V}$, $U_c = -1.00\,\mathrm{V}$. Brighter room: $U_d = 0$, $U_c = -1.50\,\mathrm{V}$.

1. The left channel alone cannot distinguish "torch on the left" from "brighter room light": both give $U_L = -1.50\,\mathrm{V}$.
2. The sign of $U_d$ gives the side (positive for torch left with the convention $U_d = U_R - U_L$). The value of $U_c$ tracks the shared brightness.
```


### Anticipate


(Task_A1_5_2)=
#### Task A1: Two converters by the numbers
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

Use the converter from Manual 4.1 with $R_f = 100\,\mathrm{k}\Omega$, so $U = -I R_f$ for each photodiode. Assume the room light gives a photocurrent of $5\,\mathrm{\mu A}$ in each photodiode, bright room light gives $15\,\mathrm{\mu A}$ in each, and a torch close to one side adds $10\,\mathrm{\mu A}$ to that side only.

1. Calculate $U_L$ and $U_R$ for: darkness, room light only, torch left, torch right, and bright room light. Check that the middle three rows reproduce the table of [Task B1](Task_B1_5_2).
2. Explain in one sentence why both outputs are negative, even though light only ever adds photocurrent. Use the virtual ground and the direction of the current through $R_f$ from Task A5 of Manual 4.1.
3. Apply the design rule from Task C4 of Manual 4.1, $|I_{in,max}| = |U_{out,max}|/R_f$, with $|U_{out,max}| = 10\,\mathrm{V}$ as a safe linear limit on $\pm 12\,\mathrm{V}$ rails. Up to what photocurrent does each converter stay linear? How much headroom does the largest current of step 1 leave?

4. Under equal illumination, the left photodiode produces $5\,\%$ more current than the right one. Keep $R_{f,R} = 100\,\mathrm{k}\Omega$. Calculate the $R_{f,L}$ that makes $U_L = U_R$. Do you increase or decrease it? Why is changing the subtractor's $R_3$ a poor way to correct this sensor mismatch?

```{admonition} Check your answer
:class: answer, dropdown
With $U = -I \cdot 100\,\mathrm{k}\Omega$, each microampere gives $-0.10\,\mathrm{V}$. Darkness: $U_L = U_R = 0$. Room light ($5\,\mathrm{\mu A}$ each): $-0.50\,\mathrm{V}$ and $-0.50\,\mathrm{V}$. Torch left ($15$ and $5\,\mathrm{\mu A}$): $-1.50\,\mathrm{V}$ and $-0.50\,\mathrm{V}$. Torch right: the mirror image. Bright room ($15\,\mathrm{\mu A}$ each): $-1.50\,\mathrm{V}$ and $-1.50\,\mathrm{V}$.

2. The photocurrent flows into the virtual ground and cannot enter the opamp input, so it must flow through $R_f$ from the output; the output side of $R_f$ therefore sits below the virtual ground at $0\,\mathrm{V}$.
3. $|I_{in,max}| = 10\,\mathrm{V} / 100\,\mathrm{k}\Omega = 100\,\mathrm{\mu A}$. The largest current in step 1 is $15\,\mathrm{\mu A}$, so you use about $1.5\,\mathrm{V}$ of the roughly $10\,\mathrm{V}$ available: plenty of headroom.
4. With $I_L = 1.05 I_R$, balance requires $R_{f,L} = 100/1.05 \approx 95.24\,\mathrm{k}\Omega$. Reduce the gain of the more sensitive left channel. Increasing the other channel's feedback resistance could also balance the products, but changes the overall scale. Changing $R_3$ alone breaks the subtractor's ratio matching and adds common-mode leakage.
```


(Task_A2_5_2)=
#### Task A2: Derive the differential amplifier gain
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

Look at the third opamp in {numref}`fig-dual-tia-subtractor`. The left output $U_L$ reaches the inverting input through $R_1$, with $R_3$ as feedback. The right output $U_R$ reaches the non-inverting input through the divider $R_2$ and $R_4$. For this task and [Task A3](Task_A3_5_2), take the bottom of $R_4$ at ground, so $U_{ref} = 0$. [Task A4](Task_A4_5_2) lifts that point.

Derive the output by superposition, reusing your Manual 4.1 results:

1. Set $U_L = 0$. The opamp is now a non-inverting amplifier for the voltage at its positive input. Write that voltage as a divider of $U_R$, then multiply by the non-inverting gain $1 + R_3/R_1$.
2. Set $U_R = 0$. The opamp is now an inverting amplifier for $U_L$. Write its contribution.
3. Add both contributions. Show that the output is proportional to $U_R - U_L$ only, with no separate $U_c$ term, when the ratios match: $R_3/R_1 = R_4/R_2$. What is $A_d$ in that case?
4. This manual uses $R_1 = R_2 = R_3 = R_4 = 10\,\mathrm{k}\Omega$. State $A_d$ and predict $U_{out}$ for the torch left and torch right rows of [Task B1](Task_B1_5_2).

```{admonition} Hint
:class: hint, dropdown
The positive input sees $U_+ = U_R \cdot R_4/(R_2 + R_4)$, because $R_2$ and $R_4$ form a voltage divider and the opamp input draws no current. Everything else is Manual 4.1: a non-inverting gain of $1 + R_3/R_1$ on $U_+$, and an inverting gain of $-R_3/R_1$ on $U_L$.
```

```{admonition} Check your answer
:class: answer, dropdown
Adding both paths gives $U_{out} = (1 + R_3/R_1)\,\dfrac{R_4}{R_2 + R_4}\,U_R - \dfrac{R_3}{R_1}\,U_L$. The condition $R_3/R_1 = R_4/R_2 = k$ makes $(1+k)\cdot k/(1+k) = k$, so both inputs get the same weight with opposite signs: $U_{out} = k\,(U_R - U_L)$ and $A_d = k = R_3/R_1$.

With all four resistors at $10\,\mathrm{k}\Omega$, $A_d = 1$. Torch left: $U_{out} = 1 \cdot (+1.00\,\mathrm{V}) = +1.00\,\mathrm{V}$. Torch right: $-1.00\,\mathrm{V}$. Equal light gives $0\,\mathrm{V}$ for any shared brightness, because only the difference survives.
```


(Task_A3_5_2)=
#### Task A3: Resistor mismatch sets the CMRR
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

Real resistors are not exactly $10\,\mathrm{k}\Omega$. Suppose $R_3$ is $1\,\%$ high, so $R_3 = 10.1\,\mathrm{k}\Omega$, while the other three are exactly $10\,\mathrm{k}\Omega$. Keep $U_{ref} = 0$.

1. Use your general formula from [Task A2](Task_A2_5_2) step 3 to write $U_{out}$ as a coefficient times $U_R$ plus a coefficient times $U_L$, with numbers.
2. Substitute $U_R = U_c + U_d/2$ and $U_L = U_c - U_d/2$. Collect the coefficients of $U_d$ and $U_c$. These are $A_d$ and $A_c$ for this imperfect amplifier. Calculate the CMRR as a ratio and in decibels.
3. The room light gets brighter, so both inputs move together from $U_c = -0.50\,\mathrm{V}$ to $U_c = -1.50\,\mathrm{V}$ with $U_d = 0$. By how much does the output move, ideally and with the $1\,\%$ error? Compare that false signal with the genuine $1.00\,\mathrm{V}$ torch signal of [Task A2](Task_A2_5_2): can it flip the arrow on its own?
4. In one or two sentences: a simulation with perfectly matched resistors can show an almost infinite CMRR. Why will your bench circuit never reach that value?

```{admonition} Check your answer
:class: answer, dropdown
1. The $U_R$ coefficient is $(1 + 10.1/10)\cdot 10/(10+10) = 2.01 \cdot 0.5 = 1.005$. The $U_L$ coefficient is $-10.1/10 = -1.01$. So $U_{out} = 1.005\,U_R - 1.01\,U_L$.
2. Collecting terms: $A_d = (1.005 + 1.01)/2 \approx 1.01$ and $A_c = 1.005 - 1.01 = -0.005$. The CMRR is $|1.01 / {-0.005}| \approx 200$, which is $20 \log_{10}(200) \approx 46\,\mathrm{dB}$.
3. Ideally the output does not move at all. With the error, $U_{out} = A_c\,U_c$, so it moves from $+2.5\,\mathrm{mV}$ at $U_c = -0.50\,\mathrm{V}$ to $+7.5\,\mathrm{mV}$ at $U_c = -1.50\,\mathrm{V}$, a $5\,\mathrm{mV}$ false shift. That is two hundred times smaller than the torch signal, so it cannot flip the arrow by itself, but it does shift the point that your software must treat as "centred".
4. The rejection relies on two resistor ratios cancelling exactly. Component tolerances make the ratios slightly unequal, and whatever mismatch remains after you pick your resistors sets the bench CMRR, no matter how ideal the opamp is.
```


(Task_A4_5_2)=
#### Task A4: Fit the difference into the ADC window
```{admonition} Estimated time: 15 min
:class: estimated-time no-content
```

The output of [Task A2](Task_A2_5_2) swings both positive and negative, but an ALPACA ADC input only accepts $0$ to about $3\,\mathrm{V}$ (see the signals intro from the Preparation list). The fix is in {numref}`fig-dual-tia-subtractor`: the bottom of $R_4$ does not go to ground but to a reference voltage $U_{ref} = 1.5\,\mathrm{V}$, the middle of the window. In the practicum you will source $U_{ref}$ from an ALPACA DAC output set to $1.5\,\mathrm{V}$; here you only need the arithmetic.

1. Quick check with matched $10\,\mathrm{k}\Omega$ resistors: if $U_L = U_R = U$, the divider puts $U_+ = (U_R + U_{ref})/2$ and the non-inverting gain is $2$. Show that $U_{out} = U_{ref}$ for any shared $U$. This is the level-shifted version of "equal light gives zero".
2. With matched ratios the full result is $U_{out} = U_{ref} + A_d\,U_d$. The detector must handle a photocurrent difference of up to $15\,\mathrm{\mu A}$ between the sides. With $R_f = 100\,\mathrm{k}\Omega$, what is the largest $|U_d|$? What is therefore the largest $A_d$ that keeps $U_{out}$ inside $0$ to $3\,\mathrm{V}$ around $U_{ref} = 1.5\,\mathrm{V}$? Confirm that the $A_d = 1$ of [Task A2](Task_A2_5_2) exactly meets this specification.
3. Convert the three key outputs to ADC counts with the conversion from the signals intro, $\mathrm{counts} = U / 3.3\,\mathrm{V} \cdot 65535$: equal light ($1.5\,\mathrm{V}$), torch fully left ($2.5\,\mathrm{V}$), torch fully right ($0.5\,\mathrm{V}$).
4. One unit of the `read_u16()` scale represents $3.3\,\mathrm{V}/65535 \approx 50\,\mathrm{\mu V}$. The Pico ADC itself has 12-bit resolution, so an ideal hardware step is about $3.3\,\mathrm{V}/4096$. Express both in photocurrent units using $A_d R_f$. Compare the hardware step with the false $2.5\,\mathrm{mV}$ offset from [Task A3](Task_A3_5_2). What does this tell you about the precision of a displayed count?

The [MicroPython RP2 reference](https://docs.micropython.org/en/latest/rp2/quickref.html#adc-analog-to-digital-conversion) describes the 12-bit ADC and its scaled `read_u16()` result. Averaging helps with noise, but the returned number alone does not establish 16-bit measurement accuracy.

```{figure} images/adc_window.svg
---
name: fig-adc-window
height: 360px
---
The level-shifted output inside the ADC window. Equal light parks the output at mid-window; the sign of the departure gives the side.
```

```{admonition} Check your answer
:class: answer, dropdown
1. With $U_L = U_R = U$: $U_{out} = 2 \cdot (U + U_{ref})/2 - U = U_{ref}$, independent of the shared level $U$.
2. The largest difference is $|U_d| = 15\,\mathrm{\mu A} \cdot 100\,\mathrm{k}\Omega = 1.5\,\mathrm{V}$. The output may depart from $U_{ref}$ by at most $1.5\,\mathrm{V}$ in either direction, so $A_d \le 1.5/1.5 = 1$. The four equal $10\,\mathrm{k}\Omega$ resistors give exactly $A_d = 1$.
3. Equal light: $1.5/3.3 \cdot 65535 \approx 29800$ counts. Torch fully left ($2.5\,\mathrm{V}$): about $49600$ counts. Torch fully right ($0.5\,\mathrm{V}$): about $9900$ counts.
4. A software-scale unit corresponds to about $0.50\,\mathrm{nA}$. An ideal hardware step is about $0.806\,\mathrm{mV}$, or $8.1\,\mathrm{nA}$ at $A_d R_f = 100\,\mathrm{k}\Omega$. The $2.5\,\mathrm{mV}$ mismatch offset is about $50$ software-scale units but only about three ideal hardware steps. Both quantisation and analogue errors matter. You measure the baseline and noise before choosing the display thresholds in [Task A5](Task_A5_5_2).
```


(Task_A5_5_2)=
#### Task A5: Five position levels from one voltage
```{admonition} Estimated time: 10 min
:class: estimated-time no-content
```

Show five levels: **far left, left, centre, right, far right**. These are bins of the relative light response. They are not five calibrated angles. Positive departures from the baseline mean light on the left, following [Task B1](Task_B1_5_2).

1. Define the **baseline** as the averaged ADC reading with both sensors under equal illumination and no directional torch. Explain why a baseline cannot remove a gain mismatch at every ambient level.
2. Let $d = \mathrm{counts} - \mathrm{baseline}$. Start with a centre deadband $D = 50\,\mathrm{mV}$ and a strong-response threshold $H = 750\,\mathrm{mV}$. At $65535/3.3$ software-scale units per volt, these are about $993$ and $14894$ units. Complete the table and sketch a horizontal display with five cells and one marker.

   | corrected reading $d$ | level | meaning |
   |---|---|---|
   | $d > H$ | $+2$ | |
   | $D < d \le H$ | $+1$ | |
   | $-D \le d \le D$ | $0$ | |
   | $-H \le d < -D$ | $-1$ | |
   | $d < -H$ | $-2$ | |

3. Convert $D$ and $H$ to photocurrent differences using $A_d R_f$. A baseline is $29850$ units. Classify readings of $30100$, $41000$, $18000$, $49000$, and $10000$ units.
4. Write pseudocode: average a block, subtract the baseline, classify it into one of five levels, and display a label or the horizontal marker. Explain why the marker moves left when $d$ grows positive.

```{admonition} Check your answer
:class: answer dropdown
The rows mean far left, left, centre, right, and far right. With $A_d R_f = 100\,\mathrm{k}\Omega$, the two thresholds correspond to $0.50\,\mathrm{\mu A}$ and $7.5\,\mathrm{\mu A}$.

The corrected readings are $+250$, $+11150$, $-11850$, $+19150$, and $-19850$. Their levels are $0$, $+1$, $-1$, $+2$, and $-2$. A positive difference means the left converter is more negative, so the subtractor output rises and the marker belongs on the left.
```


### Simulate

```{attention}
The simulations below confirm your hand calculations from Anticipate before you touch the breadboard. If a sim disagrees with your prediction, find the bug **on paper** before you build the circuit. Catching mistakes here saves a lot of debugging time at the bench.
```


## Practicum


### Implement and Investigate

Use the same photodiode type and approved dual opamps as in Manual 4.1. You need three opamps across two dual chips, two photodiodes, four matched-ratio $10\,\mathrm{k}\Omega$ resistors, one $100\,\mathrm{k}\Omega$ resistor, one $91\,\mathrm{k}\Omega$ resistor, and a $20\,\mathrm{k}\Omega$ multiturn trimmer. Add $100\,\mathrm{nF}$ decoupling from each supply pin to ground on both chips. Keep the unused opamp in a closed feedback loop with its positive input grounded, as a voltage follower.

```{attention}
Keep the ADC disconnected until [Task I3](Task_I3_5_2) verifies the output range. The converter outputs can be negative and must never connect directly to ADC2. Use a common ground for the bench supply, ALPACA, DMM and scope. Switch the supply off before changing components or moving the photodiodes. Identify the pinout of your actual chip before applying $\pm12\,\mathrm{V}$.
```

(Task_I1_5_2)=
#### Task I1: Build and balance the two sensor channels
```{admonition} Estimated time: 35 min
:class: estimated-time no-content
```

One student builds the left converter while the other builds the right. Swap channels to check the wiring and measure them.

1. Build the two current-to-voltage converters from {numref}`fig-dual-tia-subtractor`. Keep the right feedback at $100\,\mathrm{k}\Omega$. Replace the left feedback with the $91\,\mathrm{k}\Omega$ resistor and trimmer in {numref}`fig-feedback-balance`. Connect the photodiode cathodes to the inverting inputs and their anodes to ground.
2. With the supply off and one feedback-network lead disconnected, measure the left total resistance. Set it near $100\,\mathrm{k}\Omega$, then reconnect it. The required pot setting is about $9\,\mathrm{k}\Omega$, which is not necessarily half a turn or half its travel. Measure the right resistor too.
3. Power the converters. Give both sensors the same diffuse illumination, without the card divider. Record $U_L$, $U_R$, and $U_d=U_R-U_L$ at two shared light levels. Keep both converters linear. Which channel has the larger response magnitude? Predict which way to change the left feedback resistance.
4. If the imbalance is large, power off and exchange the photodiodes while leaving the resistors and opamps in place. Repeat one light condition. Does the larger response follow the sensor or the electronic channel? Restore the original sensor positions before trimming.
5. Adjust the left trimmer to reduce the difference under equal illumination. Keep the illumination fixed while making each adjustment. Check the second light level without retuning. Aim to minimise the change in $U_d$ between the two levels. A constant residual can be removed with a software baseline later. Record the two after-trim readings and the actual feedback resistance, measured with power off and one lead disconnected.
6. Explain the result with $I_L R_{f,L}=I_R R_{f,R}$. If no setting balances the channels, check orientation, lighting, saturation and the available trim range. Do not hide a large imbalance by changing $R_3$ alone.

```{figure} images/feedback-balance.svg
---
name: fig-feedback-balance
height: 320px
alt: The left feedback network has a 91 kilohm resistor in series with a 20 kilohm potentiometer. Its wiper is strapped to one outer terminal. The right feedback remains 100 kilohms.
---
Replace only the left converter's feedback resistor. The fixed resistor keeps the resistance nonzero and the wiper strap maintains a resistive path if the wiper loses contact.
```

The left feedback spans nominally $91$ to $111\,\mathrm{k}\Omega$. This range corrects modest mismatch. It does not guarantee balance for every sensor pair or lighting geometry. Keep the feedback wiring short. If the output oscillates, check the supply decoupling and feedback connections before interpreting a DC reading.


(Task_I2_5_2)=
#### Task I2: Test the subtractor before connecting the sensors
```{admonition} Estimated time: 25 min
:class: estimated-time no-content
```

One student builds the third-opamp subtractor and the other configures its test voltages. Swap roles when recording the four cases.

1. Build the four-resistor subtractor from [Task A2](Task_A2_5_2), with $U_{ref}=0$ and no ADC connection. Keep the sensor outputs disconnected from its inputs.
2. Use DAC A for $U_L$ and DAC B for $U_R$. Set their physical reference selection and software `v_ref` consistently to $3.0\,\mathrm{V}$, as in the [signals introduction](../intros/alpaca2/signals.ipynb). Use your existing `pico` connection from the programming introduction. Run the helper below on your laptop.
3. Set the four pairs $(U_L,U_R)$ to $(0.5,1.5)$, $(1.5,0.5)$, $(0.5,0.5)$, and $(1.5,1.5)$, in volts. Measure the actual two inputs and output with the DMM in each case. The nominal outputs are $+1$, $-1$, $0$, and $0\,\mathrm{V}$.
4. Calculate $A_d$ from the output change divided by the differential-input change between the first two rows. Calculate $A_c$ from the output change divided by the common-mode change between the last two rows. Then find CMRR. If the common-mode output change is below your measurement resolution, report a bound on CMRR and state the resolution you used.
5. Record the raw measurements and compare the resulting gains with [Task S1](Task_S1_5_2). Keep sensor calibration separate from this test of the subtractor's resistor ratios.

```python
@pico.task
def set_test_inputs(u_left, u_right):
    from dac import DAC
    global test_left, test_right
    test_left = DAC("A", v_ref=3000, gain=1)
    test_right = DAC("B", v_ref=3000, gain=1)
    test_left.write(u_left)
    test_right.write(u_right)

set_test_inputs(0.5, 1.5)
```


(Task_I3_5_2)=
#### Task I3: Connect the detector and check its range
```{admonition} Estimated time: 30 min
:class: estimated-time no-content
```

One student arranges the sensor pair and light source while the other watches the three outputs. Swap roles for the ambient-light check.

1. Switch the bench supply off. Disconnect both DAC test leads from the subtractor inputs. Connect $U_L$ through $R_1$ and $U_R$ through $R_2$, as in {numref}`fig-dual-tia-subtractor`. Move the bottom of $R_4$ from ground to DAC A, set to $1.5\,\mathrm{V}$. Verify the reference with the DMM while the circuit is operating. DAC B is no longer connected to the circuit.
2. Place the photodiodes side by side and add the card divider from {numref}`fig-sensor-pair-topview`. Start with the torch far enough away that both converters stay linear. Move it left, centre and right. Record $U_L$, $U_R$, $U_{ref}$ and $U_{out}$. Does left illumination make $U_{out}$ rise, as predicted?
3. Remove the directional torch and change only the shared illumination. Record $U_{out}$ at the two light levels without changing the trim. If the divider or sensor alignment spoils the balance, adjust it under equal illumination and repeat the range check.
4. Watch $U_{out}$ on the scope throughout the intended torch movement. For the ADC test, keep it between $0.1$ and $2.9\,\mathrm{V}$, leaving margin inside the $0$ to $3\,\mathrm{V}$ window. If necessary, move the light farther away. Trimming changes the channel gain, so repeat this check after every trim or gain change.
5. Only after the range check, connect $U_{out}$ to **ADC2**, the plain single-ended input on GPIO28, and connect its ground to the circuit ground. Keep the two negative converter outputs away from this input. Record the wiring check in your entry before running [Task I4](Task_I4_5_2).


(Task_I4_5_2)=
#### Task I4: Calibrate and display five levels
```{admonition} Estimated time: 25 min
:class: estimated-time no-content
```

One student runs the acquisition and records the results. The other holds the torch at reproducible positions. Swap roles halfway through the five-level test.

The code below runs on your laptop using the existing `pico` connection. The decorated functions run MicroPython on the Pico. They set the reference and average ADC2 samples before returning one result. The host classifies that result and draws the five-cell text display. This follows the [programming introduction](../intros/alpaca2/programming.ipynb); do not create a second connection to the same Pico.

1. Run the setup cell. Check DAC A again at $1.5\,\mathrm{V}$ with $R_4$ connected. Keep the output inside the range verified in [Task I3](Task_I3_5_2).
2. Remove the directional torch, illuminate both sensors equally, and run the baseline cell. It estimates noise from twenty block averages. Record the baseline, noise estimate, deadband and strong-response threshold. The deadband is at least $50\,\mathrm{mV}$ and increases if the observed fluctuations require it.
3. Run the display cell. Find a stationary torch condition for each of the five levels. Record the mean output voltage, corrected reading, level, and physical torch position. The labels describe relative detector response; do not report them as measured angles.
4. Return to equal illumination and change the room brightness without recalibrating. Does the display remain centred? Compare this with the before-trim and after-trim measurements from [Task I1](Task_I1_5_2).
5. If a level cannot be reached before the output approaches the ADC limits, record that limitation. Investigate illumination, balance and gain instead of forcing the input beyond its range. Save one sample display and your calibration values with the log book entry.

Keep the same baseline while comparing ambient conditions. Recalibrating at every condition would conceal the drift you are testing. Close the connection with `pico.close()` when all measurements and deep dives are finished.


In [ ]:
@pico.task
def setup_detector():
    from machine import ADC
    from dac import DAC
    global detector_adc, detector_reference
    detector_adc = ADC(28)
    detector_reference = DAC("A", v_ref=3000, gain=1)
    detector_reference.write(1.5)


@pico.task
def read_block(n=100, interval_us=1000):
    from time import sleep_us
    total = 0
    low = 65535
    high = 0
    for _ in range(n):
        value = detector_adc.read_u16()
        total += value
        low = min(low, value)
        high = max(high, value)
        sleep_us(interval_us)
    return total / n, low, high


setup_detector()


In [ ]:
scale = 65535 / 3.3
calibration = [read_block()[0] for _ in range(20)]
baseline = sum(calibration) / len(calibration)
noise = max(abs(value - baseline) for value in calibration)
deadband = max(round(0.050 * scale), int(3 * noise) + 1)
strong = round(0.750 * scale)
if deadband >= strong:
    raise ValueError("Reduce noise before using the five-level display.")
print("Baseline, noise, deadband, strong:", baseline, noise, deadband, strong)


In [ ]:
from time import sleep


def position_level(delta, deadband, strong):
    if delta > strong:
        return 2
    if delta > deadband:
        return 1
    if delta < -strong:
        return -2
    if delta < -deadband:
        return -1
    return 0


labels = ("far left", "left", "centre", "right", "far right")
for _ in range(200):
    mean, low, high = read_block()
    if low < 0.1 * scale or high > 2.9 * scale:
        print("Near ADC limit: move the light away and check the circuit.")
        break
    delta = mean - baseline
    level = position_level(delta, deadband, strong)
    index = 2 - level
    cells = ["."] * 5
    cells[index] = "*"
    print("[" + "|".join(cells) + "]", labels[index],
          "U =", round(mean / scale, 3), "V; corrected =", round(delta))
    sleep(0.05)


### Compare and Conclude

Complete these tables in your log book entry. Review the evidence with the other pair at your table before asking a TA for check-off.


````{admonition} Deep dive: A smoother horizontal display
:class: deep-dive dropdown

Keep the same calibrated hardware and baseline from [Task I4](Task_I4_5_2). Replace the five-bin text display with a moving marker on your laptop. Run this cell in the notebook after the acquisition cells. It uses Matplotlib and the notebook display mechanism, while `read_block()` still samples on the Pico.

```python
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xticks([-1, 0, 1], ["left", "centre", "right"])
ax.set_xlabel("Relative detector response")
ax.axvline(0, color="0.6", linewidth=1)
marker, = ax.plot([0], [0], "o", markersize=14)
plt.close(fig)
view = display(fig, display_id=True)

for _ in range(120):
    mean, low, high = read_block()
    if low < 0.1 * scale or high > 2.9 * scale:
        print("Near ADC limit: stop and check the range.")
        break
    response = -(mean - baseline) / scale
    marker.set_data([max(-1, min(1, response))], [0])
    ax.set_title("Corrected output: {:.3f} V".format((mean - baseline) / scale))
    view.update(fig)
    sleep(0.05)
```

The marker coordinate uses a $1\,\mathrm{V}$ corrected-output scale and clips the drawing at its ends. Clipping the drawing does not protect the ADC. Move the torch slowly through the five earlier conditions and save a plot. Compare visible jitter and the smallest movement you can distinguish with the five-bin display. Does a smoother-looking display contain more sensor information? Explain why it still does not measure angle without optical calibration.
````


````{admonition} Deep dive: Flicker, averaging and response time
:class: deep-dive dropdown

Keep both sensors under equal illumination. On the scope, compare the AC variation at one converter output with the variation at the subtractor output. Record the main flicker frequency, if you can resolve one, and the two amplitudes. Explain which part of the light variation the analogue circuit rejects.

Next compare `read_block(n=10)`, `read_block(n=100)`, and `read_block(n=500)`, keeping `interval_us=1000`. Measure a fresh baseline for each averaging length. For each setting, record the spread of twenty block means and the delay when the torch moves from one side to the other. Keep the optical conditions fixed during the noise measurement.

Plot noise against averaging duration and note the response delay beside each point. Explain why a block near a whole number of flicker periods can behave differently from a shorter block. Choose an averaging duration for a detector that must follow a slowly moving hand, and justify it using your two measurements.
````
